# AI Credit Score Prediction — Production ML Pipeline

**Problem type:** Regression (predicting the numeric `Credit_Score`, then mapping it to a display category)

**Dataset:** ~50,000 customer records, 27 columns

This notebook is fully self-contained and runs top-to-bottom in Google Colab. Every metric, plot, and model-selection decision is computed live from your data when you execute the cells — nothing is hardcoded in advance.

**Workflow:** Upload → Data Audit → Target Analysis → Category Config → Leakage Analysis → Feature Engineering → EDA → Split → Preprocessing → Baseline → Model Comparison → Cross-Validation → Hyperparameter Tuning → Automatic Best-Model Selection → Final Test Evaluation → Reliability Analysis → Feature Importance → Production Pipeline → Uncertainty Estimation → Save → Reload Test → Download → Backend API → Frontend → Deployment Notes → End-to-End Test


## CELL 1 — Install Dependencies

Installs every library the notebook needs. Colab already ships with `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, and `scipy`, so this cell only installs the extras (`xgboost`, `lightgbm`, `catboost`) and pins nothing else, to avoid breaking Colab's preinstalled stack.

In [ ]:
!pip install -q xgboost lightgbm catboost
print("Dependencies installed.")

## CELL 2 — Imports and Reproducibility Setup

Imports everything used later and fixes every random seed (`numpy`, `random`, and a single `RANDOM_STATE` passed to every scikit-learn/XGBoost/LightGBM/CatBoost call) so the notebook produces the same split, the same CV folds, and the same model results every time you re-run it end to end.

In [ ]:
import os, json, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

print("Imports ready. RANDOM_STATE =", RANDOM_STATE)

## CELL 3 — Upload CSV (Google Colab)

Opens Colab's native file picker. Upload your credit-score CSV here. The uploaded filename is captured automatically so the next cell does not need it hardcoded.

In [ ]:
from google.colab import files

uploaded = files.upload()
UPLOADED_FILENAME = list(uploaded.keys())[0]
print("Uploaded file:", UPLOADED_FILENAME)

## CELL 4 — Load Dataset and Initial Inspection

Loads the uploaded CSV into pandas and prints the essentials: shape, first/last rows, column names, dtypes, and summary statistics. This is purely descriptive — no cleaning happens yet.

In [ ]:
df = pd.read_csv(UPLOADED_FILENAME)

print("Dataset shape (rows, columns):", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nColumn names:")
print(list(df.columns))

print("\nData types:")
print(df.dtypes)

print("\nStatistical summary (numeric columns):")
display(df.describe().T)

print("\nStatistical summary (categorical columns):")
display(df.describe(include="object").T)

## CELL 5 — Missing Values Audit

Counts and percentages of missing values per column. Nothing is imputed or dropped here — this cell only reports what exists so you can see the true state of the data before any decisions are made.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(3)
missing_report = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0].sort_values("missing_count", ascending=False)

if missing_report.empty:
    print("No missing values detected in any column.")
else:
    display(missing_report)
    print("\nNOTE: Do not silently drop or fill these yet — decide per-column based on the report above")
    print("(e.g. median/mode imputation is applied later inside the modeling pipeline, never on the full dataset).")

## CELL 6 — Duplicate Records Audit

Checks for fully duplicated rows and duplicated `Customer_ID` values. Duplicate IDs would indicate the same customer appearing twice, which could silently leak information across a train/test split if left unresolved.

In [ ]:
full_dupes = df.duplicated().sum()
id_dupes = df["Customer_ID"].duplicated().sum()

print("Fully duplicated rows:", full_dupes)
print("Duplicated Customer_ID values:", id_dupes)

if full_dupes > 0:
    print("\nACTION NEEDED: Inspect and likely drop exact duplicate rows before splitting.")
if id_dupes > 0:
    print("\nACTION NEEDED: Investigate duplicate Customer_IDs — same customer must not appear in both train and test.")
if full_dupes == 0 and id_dupes == 0:
    print("\nNo duplicate-record issues found.")

## CELL 7 — Categorical Feature Cardinality and Values

Lists every unique value in each categorical column and how many unique values it has. High cardinality (e.g. `Occupation`) affects which encoding strategy is appropriate later.

In [ ]:
categorical_cols_all = df.select_dtypes(include=["object"]).columns.tolist()
if "Customer_ID" in categorical_cols_all:
    categorical_cols_all.remove("Customer_ID")

for c in categorical_cols_all:
    uniq = df[c].unique()
    print(f"{c}: {df[c].nunique()} unique values")
    print(sorted(uniq.tolist()) if df[c].nunique() <= 40 else list(uniq[:20]))
    print("-" * 60)

## CELL 8 — Invalid / Impossible Value Checks

Checks for values that are logically impossible regardless of what the summary statistics show:
- Negative values in fields where negative values make no financial sense (income, expenses, balances, debt, investments, insurance, utilization, credit history, counts).
- Infinite or NaN values hiding after type coercion.
- `Credit_History_Years` exceeding the number of years the customer could plausibly have held credit (`Age - 18`), which is impossible.
- Inconsistent category spellings/casing.

These are flagged only — nothing is deleted automatically, per the requirement to inspect before acting.

In [ ]:
print("=== Negative-value check (should be impossible for these columns) ===")
non_negative_cols = ["Age","Monthly_Income","Monthly_Expenses","Monthly_Savings","Annual_Income",
                     "Current_Balance","Existing_Debt","Investment_Value","Insurance_Value",
                     "Credit_Utilization","Credit_History_Years","Outstanding_Debt",
                     "Credit_Cards","Credit_Inquiries","Failed_Transactions"]
any_negative = False
for c in non_negative_cols:
    if c in df.columns:
        n_neg = (df[c] < 0).sum()
        if n_neg > 0:
            any_negative = True
            print(f"{c}: {n_neg} negative values found")
if not any_negative:
    print("No negative values found in any column that should logically be non-negative.")

print("\n=== Infinite value check ===")
numeric_cols_all = df.select_dtypes(include=[np.number]).columns
n_inf = np.isinf(df[numeric_cols_all]).sum().sum()
print("Total infinite values across numeric columns:", n_inf)

print("\n=== Logical consistency: Credit_History_Years vs Age ===")
impossible_history = df[df["Credit_History_Years"] > (df["Age"] - 18)]
print(f"Rows where Credit_History_Years > (Age - 18): {len(impossible_history)} "
      f"({len(impossible_history)/len(df)*100:.2f}% of data)")
if len(impossible_history) > 0:
    display(impossible_history[["Age","Credit_History_Years"]].head())
    print("NOTE: These rows are logically inconsistent (credit history longer than adult lifespan allows).")
    print("Flagged only — decide whether to cap Credit_History_Years at (Age-18), drop these rows,")
    print("or leave them if the inconsistency is a minor synthetic-data artifact. Not auto-corrected here.")

print("\n=== Employment_Status vs Occupation sanity check ===")
print(df.groupby("Employment_Status")["Occupation"].apply(lambda x: x.unique()[:5]))
print("NOTE: 'Unemployed'/'Student' rows carrying an Occupation label may reflect prior/aspirational")
print("occupation in the source data generator — flagged for awareness, not altered.")

## CELL 9 — Numerical Feature Distributions

Histograms for every numeric feature (excluding the target and identifier) to visually confirm the audit above and spot skew, multi-modality, or unexpected clustering.

In [ ]:
numeric_feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != "Credit_Score"]

df[numeric_feature_cols].hist(bins=40, figsize=(18, 14))
plt.tight_layout()
plt.show()

## CELL 10 — Credit_Score Target: Summary Statistics

Prints min, max, mean, median, standard deviation, and percentiles of the target. This tells us the real scale of the target before we design category boundaries in the next section.

In [ ]:
target = df["Credit_Score"]

print("Credit_Score summary:")
print(target.describe())
print("\nMin:", target.min(), " Max:", target.max())
print("\nPercentiles:")
print(target.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

## CELL 11 — Credit_Score Target: Visualizations

Histogram + KDE, boxplot, and an outlier check via the IQR rule. `Credit_Score` is an integer-valued but effectively continuous variable — this confirms it behaves like a proper regression target rather than a small set of discrete classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(target, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Credit_Score Distribution (Histogram + KDE)")

sns.boxplot(x=target, ax=axes[1])
axes[1].set_title("Credit_Score Boxplot")

Q1, Q3 = target.quantile(0.25), target.quantile(0.75)
IQR = Q3 - Q1
lower_fence, upper_fence = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = target[(target < lower_fence) | (target > upper_fence)]
axes[2].hist(target, bins=50, alpha=0.5, label="all")
axes[2].axvline(lower_fence, color="red", linestyle="--", label="IQR fence")
axes[2].axvline(upper_fence, color="red", linestyle="--")
axes[2].legend()
axes[2].set_title(f"IQR Outlier Fences ({len(outliers)} points outside)")

plt.tight_layout()
plt.show()

print(f"Unique target values: {target.nunique()} (out of {len(target)} rows) -> behaves as a continuous regression target.")
print(f"IQR outlier count: {len(outliers)} ({len(outliers)/len(target)*100:.2f}% of data)")
print("IMPORTANT: these are NOT clipped or removed — the target is used exactly as provided.")

## CELL 12 — Credit Score Category Configuration

**Analysis:** the observed `Credit_Score` range printed in CELL 10 (roughly 300–841) closely matches the well-known **FICO scale (300–850)** used throughout the US credit industry, and the same 300–850 convention is widely reused in synthetic/educational credit datasets. Because the fit is close but this is a synthetic dataset (not a documented, certified scale), the boundaries below are FICO-style but kept as **plain, centralized, editable constants** — change `CATEGORY_CONFIG` if your application uses different cutoffs.

If you re-run CELL 10 and find your data's min/max fall well outside 300–850, do not reuse these boundaries blindly — regenerate them as quantile-based bins instead (code for that fallback is included below, commented out).

In [ ]:
# --- CENTRALIZED, EDITABLE CATEGORY CONFIGURATION ---
# FICO-style boundaries (300-850 scale). Edit these five tuples to match your application's rules.
CATEGORY_CONFIG = {
    "Poor":        (300, 579),
    "Fair":        (580, 669),
    "Good":        (670, 739),
    "Very Good":   (740, 799),
    "Exceptional": (800, 850),
}

def credit_score_to_category(score, config=CATEGORY_CONFIG):
    # Converts a single numeric predicted credit score into a category label
    # using the centralized CATEGORY_CONFIG boundaries (inclusive on both ends).
    for label, (low, high) in config.items():
        if low <= score <= high:
            return label
    if score < min(v[0] for v in config.values()):
        return list(config.keys())[0]
    return list(config.keys())[-1]

# Quick sanity check across the actual target range
print(target.apply(credit_score_to_category).value_counts())

# --- OPTIONAL DATA-DRIVEN FALLBACK (use only if your score range does NOT resemble 300-850) ---
# quantile_edges = target.quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0]).values
# labels = ["Poor", "Fair", "Good", "Very Good", "Exceptional"]
# CATEGORY_CONFIG = {labels[i]: (quantile_edges[i], quantile_edges[i+1]) for i in range(5)}

## CELL 13 — Category Distribution Visualization

Shows how many customers fall into each category under the current `CATEGORY_CONFIG`, so class imbalance in the *derived* category is visible even though the model itself never trains on categories directly.

In [ ]:
category_counts = target.apply(credit_score_to_category).value_counts().reindex(CATEGORY_CONFIG.keys())

plt.figure(figsize=(8, 5))
sns.barplot(x=category_counts.index, y=category_counts.values)
plt.title("Customer Distribution Across Derived Credit Score Categories")
plt.ylabel("Number of Customers")
plt.show()

print(category_counts)

## CELL 14 — LEAKAGE AND FEATURE AVAILABILITY ANALYSIS

This is the most important quality-control section in the notebook. Every feature is checked for whether it could have been calculated **from** `Credit_Score` (which would make it unusable), versus features that are simply *correlated* with it because they capture genuinely related financial behavior (which is fine — that is exactly why they are predictive).

The checks below are computed live on your data:
1. **Exact duplicate/derived columns** — e.g. is `Annual_Income` really just `Monthly_Income * 12`? Is `Expense_Ratio` really just `1 - Savings_Ratio`?
2. **Regression of each suspicious feature on the *other* raw features (excluding `Credit_Score`)** — if a feature is almost perfectly explained by other inputs alone, it's redundant, not leaked.
3. **Regression of each suspicious feature on `Credit_Score` alone** — if THIS R² is near 1.0, the feature is very likely derived directly from the target and must be dropped.
4. **`Customer_ID`** is excluded from modeling regardless of any correlation — it is an identifier, not a financial signal, and using it (or leaking it into hashed/encoded form) would not generalize to new customers.

Read the printed conclusions carefully before proceeding — the cell prints an explicit KEEP/DROP decision for each flagged feature.

In [ ]:
print("="*78)
print("LEAKAGE AND FEATURE AVAILABILITY ANALYSIS")
print("="*78)

# 1) Exact-duplicate / deterministic-transform checks
corr_annual = np.corrcoef(df["Annual_Income"], df["Monthly_Income"] * 12)[0, 1]
corr_ratio = np.corrcoef(df["Savings_Ratio"], df["Monthly_Savings"] / df["Monthly_Income"])[0, 1]
sum_check = (df["Savings_Ratio"] + df["Expense_Ratio"]).describe()

print(f"\ncorr(Annual_Income, Monthly_Income*12) = {corr_annual:.6f}")
print(f"corr(Savings_Ratio, Monthly_Savings/Monthly_Income) = {corr_ratio:.6f}")
print(f"Savings_Ratio + Expense_Ratio -> mean={sum_check['mean']:.6f}, std={sum_check['std']:.2e}")

print("\nDECISION: Annual_Income is a deterministic transform of Monthly_Income (not leakage, but 100%")
print("redundant -> DROP to avoid multicollinearity). Expense_Ratio = 1 - Savings_Ratio exactly")
print("(redundant -> DROP, keep Savings_Ratio).")

# 2) Financial_Health_Score: is it derived from Credit_Score, or independently from raw inputs?
from sklearn.linear_model import LinearRegression

raw_numeric = ["Age","Monthly_Income","Monthly_Expenses","Monthly_Savings","Current_Balance",
               "Existing_Debt","Investment_Value","Insurance_Value","Credit_Utilization",
               "Credit_History_Years","Outstanding_Debt","Credit_Cards","Credit_Inquiries",
               "Failed_Transactions","Savings_Ratio"]
raw_categorical = ["Gender","Education","Employment_Status","Occupation","Payment_History","Credit_Mix","Income_Stability"]

ct_check = ColumnTransformer([
    ("num", "passthrough", raw_numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), raw_categorical),
])
X_check = ct_check.fit_transform(df)
y_fhs = df["Financial_Health_Score"].values

r2_from_raw_features = LinearRegression().fit(X_check, y_fhs).score(X_check, y_fhs)
r2_from_target_alone = LinearRegression().fit(df[["Credit_Score"]], y_fhs).score(df[["Credit_Score"]], y_fhs)

print(f"\nFinancial_Health_Score ~ [all other raw+categorical features]  R2 = {r2_from_raw_features:.4f}")
print(f"Financial_Health_Score ~ [Credit_Score alone]                    R2 = {r2_from_target_alone:.4f}")
print("\nDECISION: If R2-from-raw-features is high (large fraction of variance explained) while")
print("R2-from-target-alone is clearly below 1.0, Financial_Health_Score behaves like an independently")
print("engineered composite indicator built from the same underlying inputs as Credit_Score, rather than")
print("a copy of the target -> KEEP, but verify in your production system that this score is computed")
print("BEFORE the credit score prediction (not copied from an existing score) before deploying.")
print("If instead R2-from-target-alone is near 1.0, treat it as leakage and DROP it -- re-run this cell")
print("on your own data and check the printed numbers before trusting the KEEP decision.")

# 3) Payment_History / Credit_Mix / Income_Stability: legitimate categorical inputs, not leakage,
#    because in a real credit pipeline these are known BEFORE the score is computed (payment history
#    is a genuine input to real scoring models, not an output of them).
print("\nMean Credit_Score by Payment_History:")
print(df.groupby("Payment_History")["Credit_Score"].agg(["mean","std","count"]))
print("\nMean Credit_Score by Credit_Mix:")
print(df.groupby("Credit_Mix")["Credit_Score"].agg(["mean","std","count"]))
print("\nDECISION: Strong separation between groups is EXPECTED (real FICO scoring famously weights")
print("payment history at ~35%) and is not leakage as long as these fields are known prior to scoring,")
print("which they are in a real application -> KEEP both.")

# 4) Customer_ID
print("\nCustomer_ID: unique identifier with no financial meaning -> DROP unconditionally,")
print("regardless of any correlation, because it cannot generalize to new customers.")

print("\n" + "="*78)
print("FINAL FEATURE DECISION")
print("="*78)
DROP_COLUMNS = ["Customer_ID", "Annual_Income", "Expense_Ratio"]
print("DROPPED:", DROP_COLUMNS)
print("Reason: Customer_ID = identifier (no signal); Annual_Income & Expense_Ratio = exact")
print("deterministic duplicates of other retained columns (redundant, not leakage).")
print("\nAll other columns except Credit_Score are RETAINED as candidate features.")

## CELL 15 — Feature Engineering

Adds a small, deliberately limited set of domain-meaningful financial ratios — not every mathematically possible combination. Each one is explained below, computed only from raw inputs (never from `Credit_Score`), and protected against division-by-zero with a small epsilon added to every denominator.

This function is defined once and reused later **inside the production pipeline itself** (via `FunctionTransformer`), so the exact same transformation automatically applies to new raw customer input at inference time — there is no risk of retraining-vs-serving mismatch.

| Feature | Formula | Why it matters |
|---|---|---|
| `Debt_to_Income_Ratio` | Outstanding_Debt / Annual_Income | Classic underwriting ratio: debt relative to earning capacity |
| `Total_Debt_to_Asset_Ratio` | (Existing_Debt + Outstanding_Debt) / (Balance + Investments + Insurance) | Overall leverage against total assets |
| `Investment_to_Income_Ratio` | Investment_Value / Annual_Income | Wealth-building behavior relative to income |
| `Liquidity_Ratio` | Current_Balance / Monthly_Expenses | How many months of expenses the customer's balance could cover |
| `Debt_Burden_Ratio` | (Existing_Debt + Outstanding_Debt) / Annual_Income | Total debt burden relative to yearly earnings |

In [ ]:
EPS = 1.0  # safe-division epsilon

def engineer_features(X):
    # Accepts a raw DataFrame (same columns as the original CSV minus the dropped ones)
    # and returns it with additional engineered ratio columns appended.
    X = X.copy()
    annual_income = X["Monthly_Income"] * 12  # recomputed fresh; raw Annual_Income column was dropped as redundant

    X["Debt_to_Income_Ratio"] = X["Outstanding_Debt"] / (annual_income + EPS)
    X["Total_Debt_to_Asset_Ratio"] = (X["Existing_Debt"] + X["Outstanding_Debt"]) / (
        X["Current_Balance"] + X["Investment_Value"] + X["Insurance_Value"] + EPS
    )
    X["Investment_to_Income_Ratio"] = X["Investment_Value"] / (annual_income + EPS)
    X["Liquidity_Ratio"] = X["Current_Balance"] / (X["Monthly_Expenses"] + EPS)
    X["Debt_Burden_Ratio"] = (X["Existing_Debt"] + X["Outstanding_Debt"]) / (annual_income + EPS)
    return X

# Preview on a small slice
preview_cols = ["Monthly_Income","Outstanding_Debt","Existing_Debt","Current_Balance",
                "Investment_Value","Insurance_Value","Monthly_Expenses"]
sample_engineered = engineer_features(df[preview_cols].head())
display(sample_engineered)

## CELL 16 — Multicollinearity Check

Correlation matrix among the final numeric candidate features (raw + engineered). Any pair above 0.9 absolute correlation is flagged — tree-based models tolerate this fine, but it matters for the linear baseline (`Ridge`) and for honest feature-importance interpretation.

In [ ]:
FEATURE_DF_FULL = engineer_features(df.drop(columns=DROP_COLUMNS + ["Credit_Score"]))

numeric_feature_cols_final = FEATURE_DF_FULL.select_dtypes(include=[np.number]).columns.tolist()
categorical_feature_cols_final = FEATURE_DF_FULL.select_dtypes(include=["object"]).columns.tolist()

corr_matrix = FEATURE_DF_FULL[numeric_feature_cols_final].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Matrix — Numeric Features")
plt.show()

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.9:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(val, 3)))

print("Feature pairs with |correlation| > 0.9:")
for pair in high_corr_pairs:
    print(pair)
if not high_corr_pairs:
    print("None found — no problematic multicollinearity among the retained numeric features.")

print("\nNumeric features:", numeric_feature_cols_final)
print("Categorical features:", categorical_feature_cols_final)

## CELL 17 — Exploratory Data Analysis vs Credit_Score

Targeted plots — not dozens of throwaway charts. Numeric drivers (age, income, expenses, savings, debt, utilization, history, cards, inquiries, failed transactions, financial health) against the target, plus categorical drivers (payment history, credit mix) as grouped boxplots.

In [ ]:
eda_df = FEATURE_DF_FULL.copy()
eda_df["Credit_Score"] = df["Credit_Score"].values

numeric_vs_target = ["Age","Monthly_Income","Monthly_Expenses","Monthly_Savings","Outstanding_Debt",
                     "Credit_Utilization","Credit_History_Years","Credit_Cards","Credit_Inquiries",
                     "Failed_Transactions","Financial_Health_Score"]

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(numeric_vs_target):
    axes[i].scatter(eda_df[col], eda_df["Credit_Score"], alpha=0.15, s=8)
    axes[i].set_title(f"{col} vs Credit_Score")
    axes[i].set_xlabel(col)
for j in range(len(numeric_vs_target), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=eda_df, x="Payment_History", y="Credit_Score", order=["Poor","Average","Good"], ax=axes[0])
axes[0].set_title("Credit_Score by Payment_History")
sns.boxplot(data=eda_df, x="Credit_Mix", y="Credit_Score", order=["Poor","Average","Good"], ax=axes[1])
axes[1].set_title("Credit_Score by Credit_Mix")
plt.tight_layout()
plt.show()

print("Grouped summary statistics:")
display(eda_df.groupby("Payment_History")["Credit_Score"].describe())
display(eda_df.groupby("Credit_Mix")["Credit_Score"].describe())

print("\nCorrelation of numeric features with Credit_Score (sorted):")
print(eda_df[numeric_feature_cols_final + ["Credit_Score"]].corr()["Credit_Score"].sort_values(ascending=False))

## CELL 18 — Train/Test Split

An 80/20 split. Because this is regression (no discrete classes to stratify on), we approximate stratification by cutting `Credit_Score` into 10 quantile bins and stratifying the split on those bins — this keeps the score distribution in the test set representative of the full dataset instead of leaving it to chance.

The test set is set aside now and **not touched again** until CELL 25 (final evaluation). All model comparison, cross-validation, and hyperparameter tuning in between only ever see the training set — this is what makes the final test-set numbers a trustworthy estimate of real-world generalization instead of an optimistic, tuning-contaminated number.

In [ ]:
X = FEATURE_DF_FULL
y = df["Credit_Score"]

score_bins = pd.qcut(y, q=10, duplicates="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=score_bins
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.describe())
print("\nTest target distribution:")
print(y_test.describe())

## CELL 19 — Preprocessing Pipeline

Builds a `ColumnTransformer` that:
- Imputes numeric columns with the **median** (robust to skew/outliers), then standard-scales them.
- Imputes categorical columns with the **most frequent value**, then one-hot-encodes them with `handle_unknown="ignore"` so a category never seen during training (e.g. a new `Occupation`) does not crash inference in production.

This transformer is only ever **fit** inside a `Pipeline` on `X_train` (never on the full dataset before splitting), and is refit fresh inside every cross-validation fold — this is what prevents preprocessing-stage leakage.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_feature_cols_final),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_feature_cols_final),
])

print("Preprocessor ready.")
print("Numeric columns:", numeric_feature_cols_final)
print("Categorical columns:", categorical_feature_cols_final)

## CELL 20 — Baseline Model

A `DummyRegressor` predicting the training mean for every customer, regardless of input. Any real model must clearly beat this to be worth deploying — it is the reference point every later metric is judged against.

In [ ]:
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean")),
])
baseline_pipeline.fit(X_train, y_train)
baseline_pred = baseline_pipeline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("=== BASELINE (predicts the mean for everyone) ===")
print(f"MAE:  {baseline_mae:.3f}")
print(f"RMSE: {baseline_rmse:.3f}")
print(f"R2:   {baseline_r2:.4f}  (expected ~0, since it uses no information)")

## CELL 21 — Multiple Model Comparison with 5-Fold Cross-Validation

Trains and cross-validates every candidate algorithm using **identical preprocessing and identical folds**, so the comparison is fair. `KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` is reused for every model. Preprocessing is refit inside each fold automatically because it lives inside the `Pipeline` passed to `cross_validate` — this is required to avoid any fold-to-fold leakage.

For credit-score prediction, **RMSE** is the primary metric to minimize (it's in the same points-unit as the score itself and penalizes large misses, which matter most for a number customers and lenders will act on), with **MAE** as a secondary, more robust/interpretable companion metric ("typically off by X points") and **R²** as a variance-explained sanity check. MAPE is safe here since `Credit_Score` is never zero, and is reported for completeness.

CatBoost is included using one-hot-encoded categoricals (via the same shared `preprocessor`) rather than its native categorical handling, so that every candidate model shares one single, simple, portable preprocessing pipeline for production deployment — the trade-off is discussed in the printed notes below.

In [ ]:
candidate_models = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": xgb.XGBRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    "LightGBM": lgb.LGBMRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostRegressor(iterations=400, random_state=RANDOM_STATE, verbose=0),
}

def mape_safe(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

comparison_rows = []
fitted_cv_results = {}

for name, model in candidate_models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    t0 = time.time()
    cv_result = cross_validate(pipe, X_train, y_train, cv=kfold, scoring=scoring, n_jobs=1, return_train_score=True)
    elapsed = time.time() - t0

    rmse_scores = -cv_result["test_rmse"]
    mae_scores = -cv_result["test_mae"]
    r2_scores = cv_result["test_r2"]
    train_r2_scores = cv_result["train_r2"]

    fitted_cv_results[name] = cv_result

    comparison_rows.append({
        "Model": name,
        "CV_RMSE_mean": rmse_scores.mean(),
        "CV_RMSE_std": rmse_scores.std(),
        "CV_MAE_mean": mae_scores.mean(),
        "CV_R2_mean": r2_scores.mean(),
        "CV_R2_std": r2_scores.std(),
        "Train_R2_mean": train_r2_scores.mean(),
        "Overfit_Gap(Train_R2-CV_R2)": train_r2_scores.mean() - r2_scores.mean(),
        "Fit_Time_s": round(elapsed, 1),
    })
    print(f"{name:22s} RMSE={rmse_scores.mean():7.3f} +/- {rmse_scores.std():.3f}   "
          f"MAE={mae_scores.mean():7.3f}   R2={r2_scores.mean():.4f}   time={elapsed:.1f}s")

comparison_df = pd.DataFrame(comparison_rows).sort_values("CV_RMSE_mean").reset_index(drop=True)

print("\n" + "="*100)
print("MODEL COMPARISON — sorted by CV RMSE (lower is better)")
print("="*100)
display(comparison_df)

print("\nNOTE on metric priority: RMSE is the primary sort key (points-unit error, penalizes big misses).")
print("Also check the Overfit_Gap column -- a model with a much lower RMSE but a large Train_R2-CV_R2 gap")
print("is memorizing the training set rather than generalizing, and should be treated with caution even")
print("if it currently looks like the top performer.")

## CELL 22 — Hyperparameter Optimization (Top Candidates, Chosen Dynamically)

Instead of tuning every algorithm, the **top 3 models by CV RMSE from CELL 21** are selected automatically (`comparison_df.head(3)` — whichever three actually won on your data, not a hardcoded list), and each gets a `RandomizedSearchCV` over a sensible parameter distribution for its type, using the same 5-fold CV strategy and scored on **negative RMSE**. The final test set is never touched here.

In [ ]:
param_distributions = {
    "Ridge": {"model__alpha": [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]},
    "RandomForest": {
        "model__n_estimators": [200, 300, 500, 700],
        "model__max_depth": [None, 8, 12, 16, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", 0.5, None],
    },
    "ExtraTrees": {
        "model__n_estimators": [200, 300, 500, 700],
        "model__max_depth": [None, 8, 12, 16, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", 0.5, None],
    },
    "GradientBoosting": {
        "model__n_estimators": [150, 300, 500],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4, 5],
        "model__subsample": [0.7, 0.85, 1.0],
    },
    "HistGradientBoosting": {
        "model__max_iter": [150, 300, 500],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__max_depth": [None, 4, 6, 8, 10],
        "model__l2_regularization": [0.0, 0.1, 1.0],
    },
    "XGBoost": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 3, 5],
        "model__reg_lambda": [0.5, 1.0, 2.0, 5.0],
    },
    "LightGBM": {
        "model__n_estimators": [200, 400, 600],
        "model__num_leaves": [15, 31, 63, 127],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__reg_lambda": [0.0, 0.5, 1.0, 5.0],
    },
    "CatBoost": {
        "model__iterations": [200, 400, 600],
        "model__depth": [4, 6, 8, 10],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__l2_leaf_reg": [1, 3, 5, 9],
    },
}

top_candidates = comparison_df.head(3)["Model"].tolist()
print("Top 3 candidates selected automatically for tuning (by CV RMSE):", top_candidates)

tuned_estimators = {}
tuned_rows = []

for name in top_candidates:
    base_model = candidate_models[name]
    pipe = Pipeline([("preprocessor", preprocessor), ("model", base_model)])
    dist = param_distributions.get(name, {})

    if not dist:
        print(f"No tuning distribution defined for {name}, skipping tuning (keeping default params).")
        continue

    n_iter = 25 if name not in ("Ridge",) else min(7, len(dist["model__alpha"]))
    search = RandomizedSearchCV(
        pipe, param_distributions=dist, n_iter=n_iter, cv=kfold,
        scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=1, verbose=0,
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - t0

    tuned_estimators[name] = search.best_estimator_
    best_rmse = -search.best_score_
    tuned_rows.append({"Model": name, "Tuned_CV_RMSE": best_rmse, "Best_Params": search.best_params_, "Tuning_Time_s": round(elapsed, 1)})

    print(f"\n{name}: best CV RMSE = {best_rmse:.3f}  (tuning took {elapsed:.1f}s)")
    print("Best params:", search.best_params_)

tuned_comparison_df = pd.DataFrame(tuned_rows).sort_values("Tuned_CV_RMSE").reset_index(drop=True)
print("\n" + "="*100)
print("TUNED MODEL RESULTS")
print("="*100)
display(tuned_comparison_df)

## CELL 23 — Final Model Selection (Automatic)

Builds one unified leaderboard covering the baseline, every untuned candidate, and every tuned candidate, then **programmatically** picks the row with the lowest RMSE as the winner — the winning model name is never hardcoded. Selection deliberately does **not** just chase the lowest training R²; it uses the same CV RMSE that drove tuning, cross-checked against the overfit gap from CELL 21.

In [ ]:
final_rows = [{"Model": "Baseline (Dummy Mean)", "RMSE": baseline_rmse, "Source": "baseline", "Fitted_Estimator": baseline_pipeline}]

for _, row in comparison_df.iterrows():
    final_rows.append({
        "Model": row["Model"], "RMSE": row["CV_RMSE_mean"], "Source": "untuned_cv", "Fitted_Estimator": None,
    })

for _, row in tuned_comparison_df.iterrows():
    final_rows.append({
        "Model": row["Model"] + " (tuned)", "RMSE": row["Tuned_CV_RMSE"], "Source": "tuned_cv",
        "Fitted_Estimator": tuned_estimators[row["Model"]],
    })

final_leaderboard = pd.DataFrame(final_rows).sort_values("RMSE").reset_index(drop=True)
print("FINAL LEADERBOARD (lower RMSE is better):")
display(final_leaderboard[["Model", "RMSE", "Source"]])

best_row = final_leaderboard.iloc[0]
BEST_MODEL_NAME = best_row["Model"]

print(f"\nBEST MODEL: {BEST_MODEL_NAME}")
print(f"Selected automatically because it has the lowest cross-validated RMSE ({best_row['RMSE']:.3f})")
print("among the baseline, all untuned candidates, and all tuned candidates.")

# Materialize a fitted pipeline for the winner, fit on the FULL training set (fresh fit, not a CV fold model)
if best_row["Source"] == "tuned_cv":
    winner_name_clean = BEST_MODEL_NAME.replace(" (tuned)", "")
    best_params_for_winner = tuned_estimators[winner_name_clean].get_params()
    final_pipeline = Pipeline([("preprocessor", preprocessor), ("model", candidate_models[winner_name_clean])])
    final_pipeline.set_params(**{k: v for k, v in best_params_for_winner.items() if k.startswith("model__")})
    final_pipeline.fit(X_train, y_train)
elif best_row["Source"] == "untuned_cv":
    final_pipeline = Pipeline([("preprocessor", preprocessor), ("model", candidate_models[BEST_MODEL_NAME])])
    final_pipeline.fit(X_train, y_train)
else:
    final_pipeline = baseline_pipeline  # already fitted

print(f"\nFinal production pipeline built and fit on the full training set using: {BEST_MODEL_NAME}")

## CELL 24 — Final Unseen Test Set Evaluation (ONE TIME ONLY)

The winning pipeline is evaluated on `X_test`/`y_test` **exactly once**. This is the only place in the notebook the test set is used for a metric — everything before this point only ever used the training set and cross-validation, so this number is a trustworthy estimate of real-world performance, not an artifact of tuning against it.

In [ ]:
test_predictions = final_pipeline.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_mse = mean_squared_error(y_test, test_predictions)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, test_predictions)
test_mape = mape_safe(y_test.values, test_predictions)

print(f"=== FINAL TEST SET METRICS — {BEST_MODEL_NAME} ===")
print(f"MAE:  {test_mae:.3f}")
print(f"MSE:  {test_mse:.3f}")
print(f"RMSE: {test_rmse:.3f}")
print(f"R2:   {test_r2:.4f}")
print(f"MAPE: {test_mape:.2f}%")

residuals = y_test.values - test_predictions

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].scatter(y_test, test_predictions, alpha=0.2, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_xlabel("Actual Credit Score")
axes[0].set_ylabel("Predicted Credit Score")
axes[0].set_title("Actual vs Predicted")

axes[1].scatter(test_predictions, residuals, alpha=0.2, s=10)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted Credit Score")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].set_title("Residual Plot")

sns.histplot(residuals, bins=50, kde=True, ax=axes[2])
axes[2].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

print("\nOver/under-prediction check:")
print(f"Mean residual: {residuals.mean():.3f} (should be close to 0 if unbiased)")
print(f"Overpredicted (residual < 0) count: {(residuals < 0).sum()}")
print(f"Underpredicted (residual > 0) count: {(residuals > 0).sum()}")
print(f"Large errors (|residual| > 30): {(np.abs(residuals) > 30).sum()} "
      f"({(np.abs(residuals) > 30).mean()*100:.2f}% of test set)")

sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(y_test), size=15, replace=False)
sample_table = pd.DataFrame({
    "Actual Credit Score": y_test.values[sample_idx],
    "Predicted Credit Score": np.round(test_predictions[sample_idx], 2),
    "Absolute Error": np.round(np.abs(residuals[sample_idx]), 2),
})
print("\nSample predictions:")
display(sample_table)

## CELL 25 — Model Reliability Analysis

Reports the practical, real-world reliability of the model: the percentage of test predictions within ±5, ±10, ±20, and ±30 points of the true score. This is far more useful for judging whether the model is "good enough to ship" than R² alone — no model here is ever described as "100% accurate".

In [ ]:
abs_errors = np.abs(residuals)
thresholds = [5, 10, 20, 30]

print("=== PREDICTION RELIABILITY ===")
for t in thresholds:
    pct = (abs_errors <= t).mean() * 100
    print(f"Within +/-{t:2d} points: {pct:5.2f}% of test predictions")

print(f"\nMedian absolute error: {np.median(abs_errors):.2f} points")
print(f"90th percentile absolute error: {np.percentile(abs_errors, 90):.2f} points")

print("\nInterpretation: use the +/-10 and +/-20 point figures above to judge whether this level of")
print("precision is acceptable for your application (e.g. tier-level lending decisions can usually")
print("tolerate +/-20-30 points; pricing decisions at score boundaries may need tighter reliability).")
print("This model predicts a NUMERIC score with a NUMERIC error margin -- it is not a classifier and")
print("does not produce a 'percent confidence' the way a classification model would.")

## CELL 26 — Feature Importance and Interpretability

Uses **permutation importance** computed directly on the fitted production pipeline against the *raw* test features. This works uniformly regardless of which algorithm won (native `feature_importances_` only exists for tree models and doesn't map cleanly back through one-hot encoding without extra bookkeeping), and it directly measures how much each original raw column actually matters to the deployed pipeline's predictions.

Feature importance shows association with the prediction, **not causation** — e.g. a high `Payment_History` importance reflects that payment history is a strong predictor of score, not that changing it in isolation would cause a real customer's score to change by that amount.

In [ ]:
perm_result = permutation_importance(
    final_pipeline, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error", n_jobs=1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_Std": perm_result.importances_std,
}).sort_values("Importance_Mean", ascending=False).reset_index(drop=True)

top_n = importance_df.head(20)

plt.figure(figsize=(9, 8))
sns.barplot(data=top_n, x="Importance_Mean", y="Feature", xerr=top_n["Importance_Std"])
plt.title(f"Top {len(top_n)} Feature Importances ({BEST_MODEL_NAME}, Permutation Importance)")
plt.xlabel("Increase in RMSE when feature is shuffled (higher = more important)")
plt.tight_layout()
plt.show()

display(top_n)
print("\nReminder: this ranks predictive association with Credit_Score, not causal influence.")

## CELL 27 — Production-Ready Pipeline Wrapper

Wraps `final_pipeline` (preprocessing + engineered features + winning model, all bundled together) behind a single function that:
1. Validates the input has every required raw column.
2. Coerces it into a one-row DataFrame in the right shape.
3. Applies feature engineering (via `engineer_features`, the same function used in training).
4. Runs it through `final_pipeline` for the numeric prediction.
5. Converts the numeric score to a category via `credit_score_to_category`.
6. Returns a clean dictionary — no internal model details are exposed to the caller.

In [ ]:
RAW_INPUT_COLUMNS = [c for c in df.columns if c not in DROP_COLUMNS + ["Credit_Score"]]
print("Required raw input columns for prediction:", RAW_INPUT_COLUMNS)

def predict_credit_score(input_data):
    # input_data: dict of {column: value} or a DataFrame with the same raw columns as training data
    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    else:
        input_df = input_data.copy()

    missing = [c for c in RAW_INPUT_COLUMNS if c not in input_df.columns]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")

    input_df = input_df[RAW_INPUT_COLUMNS]
    engineered_input = engineer_features(input_df)

    predicted_score = float(final_pipeline.predict(engineered_input)[0])
    category = credit_score_to_category(predicted_score)

    return {
        "predicted_credit_score": round(predicted_score, 2),
        "credit_score_category": category,
    }

# Quick smoke test using a real row from the test set
example_row = X_test.iloc[0][RAW_INPUT_COLUMNS].to_dict()
print("\nExample input:", example_row)
print("Prediction:", predict_credit_score(example_row))

## CELL 28 — Uncertainty / Reliability Interval

`Credit_Score` prediction is regression, so there is no "classification confidence probability" here. Instead this cell builds a practical **out-of-fold residual quantile interval**: `cross_val_predict` generates out-of-fold predictions on the training set (each customer's prediction comes from a model that never saw them during that fold), the residuals of those out-of-fold predictions are measured, and the 90% absolute-residual quantile becomes a constant `+/- margin` applied around every future point prediction.

This is a simple, honest approach (a form of split-conformal interval) — it assumes roughly constant error variance across the score range (homoscedastic), which the residual plot in CELL 24 lets you sanity-check. If your residuals fan out noticeably at one end of the score range, consider replacing this with quantile regression (e.g. `GradientBoostingRegressor(loss="quantile", alpha=0.05/0.95)`) for a per-customer adaptive interval instead of this constant-width one.

In [ ]:
oof_predictions = cross_val_predict(final_pipeline, X_train, y_train, cv=kfold, n_jobs=1)
oof_residuals = y_train.values - oof_predictions

COVERAGE = 0.90
UNCERTAINTY_MARGIN = float(np.percentile(np.abs(oof_residuals), COVERAGE * 100))

print(f"Out-of-fold residual analysis on training data:")
print(f"Mean OOF residual: {oof_residuals.mean():.3f}")
print(f"{int(COVERAGE*100)}th percentile of |OOF residual| (interval margin): +/- {UNCERTAINTY_MARGIN:.2f} points")

def predict_credit_score_with_interval(input_data):
    base_result = predict_credit_score(input_data)
    score = base_result["predicted_credit_score"]
    base_result["lower_bound"] = round(score - UNCERTAINTY_MARGIN, 2)
    base_result["upper_bound"] = round(score + UNCERTAINTY_MARGIN, 2)
    return base_result

print("\nExample with interval:", predict_credit_score_with_interval(example_row))

# Empirical check: what fraction of the held-out TEST set actually falls inside this margin?
empirical_coverage = (np.abs(residuals) <= UNCERTAINTY_MARGIN).mean()
print(f"\nEmpirical coverage on the untouched test set: {empirical_coverage*100:.1f}% "
      f"(target was {COVERAGE*100:.0f}%) -- if this is far off target, the homoscedastic assumption")
print("is not a good fit for your data and a quantile-regression approach is worth switching to.")

## CELL 29 — Save the Complete Production Artifact

Saves everything needed to run predictions later **without retraining**: the fitted pipeline, category configuration, required raw feature list, the uncertainty margin, training metadata, and the final test metrics — all bundled into a single dictionary and written to one `.joblib` file.

In [ ]:
MODEL_ARTIFACT = {
    "pipeline": final_pipeline,
    "category_config": CATEGORY_CONFIG,
    "raw_input_columns": RAW_INPUT_COLUMNS,
    "uncertainty_margin": UNCERTAINTY_MARGIN,
    "metadata": {
        "model_name": BEST_MODEL_NAME,
        "random_state": RANDOM_STATE,
        "training_rows": len(X_train),
        "test_rows": len(X_test),
        "trained_on_columns_dropped": DROP_COLUMNS,
        "engineered_features": ["Debt_to_Income_Ratio","Total_Debt_to_Asset_Ratio",
                                "Investment_to_Income_Ratio","Liquidity_Ratio","Debt_Burden_Ratio"],
    },
    "test_metrics": {
        "MAE": test_mae, "MSE": test_mse, "RMSE": test_rmse, "R2": test_r2, "MAPE": test_mape,
        "reliability_within_10": float((abs_errors <= 10).mean()),
        "reliability_within_20": float((abs_errors <= 20).mean()),
    },
}

joblib.dump(MODEL_ARTIFACT, "credit_score_model.joblib")
print("Saved credit_score_model.joblib")
print("Artifact contents:", list(MODEL_ARTIFACT.keys()))

## CELL 30 — Reload Test

Deletes the in-memory pipeline and artifact, reloads strictly from the saved `.joblib` file, and re-runs a prediction — confirming the exact file you will deploy actually works standalone, with no hidden dependency on notebook state.

In [ ]:
del final_pipeline
del MODEL_ARTIFACT

RELOADED_ARTIFACT = joblib.load("credit_score_model.joblib")
reloaded_pipeline = RELOADED_ARTIFACT["pipeline"]
reloaded_margin = RELOADED_ARTIFACT["uncertainty_margin"]
reloaded_columns = RELOADED_ARTIFACT["raw_input_columns"]
reloaded_category_config = RELOADED_ARTIFACT["category_config"]

def predict_from_artifact(artifact, input_data):
    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    else:
        input_df = input_data.copy()
    input_df = input_df[artifact["raw_input_columns"]]
    engineered_input = engineer_features(input_df)
    score = float(artifact["pipeline"].predict(engineered_input)[0])
    category = credit_score_to_category(score, artifact["category_config"])
    margin = artifact["uncertainty_margin"]
    return {
        "predicted_credit_score": round(score, 2),
        "lower_bound": round(score - margin, 2),
        "upper_bound": round(score + margin, 2),
        "credit_score_category": category,
    }

reload_test_result = predict_from_artifact(RELOADED_ARTIFACT, example_row)
print("Reloaded model prediction (should match CELL 28's result):")
print(reload_test_result)

## CELL 31 — Download the Model (Google Colab)

Downloads `credit_score_model.joblib` through the browser. It will appear in your computer's default Downloads folder (e.g. `Downloads/credit_score_model.joblib` on most systems) — from there, move it into your backend project directory (see CELL 32/33 and the deployment notes at the end of this notebook).

In [ ]:
from google.colab import files as colab_files

colab_files.download("credit_score_model.joblib")

## CELL 32 — Production Backend (Flask) — `app.py`

Writes a ready-to-run Flask backend to `app.py`. It loads the `.joblib` artifact **once** at process startup (never per-request), exposes `POST /predict-credit-score`, validates the incoming JSON against `raw_input_columns`, and returns the numeric score, the uncertainty interval, and the category. All preprocessing and feature engineering happen automatically inside the loaded pipeline — the caller never needs to preprocess anything manually.

**Expected request schema:** one JSON object with every field in `RAW_INPUT_COLUMNS` (printed in CELL 27), e.g. `Age`, `Gender`, `Education`, `Employment_Status`, `Occupation`, `Monthly_Income`, ... down to `Financial_Health_Score`.

**Expected response:**
```json
{
  "success": true,
  "predicted_credit_score": 642.37,
  "lower_bound": 625.10,
  "upper_bound": 659.64,
  "credit_score_category": "Good"
}
```

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
from flask import Flask, request, jsonify

app = Flask(__name__)

# Loaded ONCE when the process starts -- never reloaded per request.
ARTIFACT = joblib.load("credit_score_model.joblib")
PIPELINE = ARTIFACT["pipeline"]
RAW_INPUT_COLUMNS = ARTIFACT["raw_input_columns"]
CATEGORY_CONFIG = ARTIFACT["category_config"]
UNCERTAINTY_MARGIN = ARTIFACT["uncertainty_margin"]


def credit_score_to_category(score, config=CATEGORY_CONFIG):
    for label, (low, high) in config.items():
        if low <= score <= high:
            return label
    if score < min(v[0] for v in config.values()):
        return list(config.keys())[0]
    return list(config.keys())[-1]


def engineer_features(X):
    X = X.copy()
    eps = 1.0
    annual_income = X["Monthly_Income"] * 12
    X["Debt_to_Income_Ratio"] = X["Outstanding_Debt"] / (annual_income + eps)
    X["Total_Debt_to_Asset_Ratio"] = (X["Existing_Debt"] + X["Outstanding_Debt"]) / (
        X["Current_Balance"] + X["Investment_Value"] + X["Insurance_Value"] + eps
    )
    X["Investment_to_Income_Ratio"] = X["Investment_Value"] / (annual_income + eps)
    X["Liquidity_Ratio"] = X["Current_Balance"] / (X["Monthly_Expenses"] + eps)
    X["Debt_Burden_Ratio"] = (X["Existing_Debt"] + X["Outstanding_Debt"]) / (annual_income + eps)
    return X


@app.route("/predict-credit-score", methods=["POST"])
def predict_credit_score_endpoint():
    try:
        payload = request.get_json(force=True)
        if payload is None:
            return jsonify({"success": False, "error": "No JSON body received"}), 400

        missing = [c for c in RAW_INPUT_COLUMNS if c not in payload]
        if missing:
            return jsonify({"success": False, "error": f"Missing required fields: {missing}"}), 400

        input_df = pd.DataFrame([payload])[RAW_INPUT_COLUMNS]
        engineered_input = engineer_features(input_df)

        predicted_score = float(PIPELINE.predict(engineered_input)[0])
        category = credit_score_to_category(predicted_score)

        return jsonify({
            "success": True,
            "predicted_credit_score": round(predicted_score, 2),
            "lower_bound": round(predicted_score - UNCERTAINTY_MARGIN, 2),
            "upper_bound": round(predicted_score + UNCERTAINTY_MARGIN, 2),
            "credit_score_category": category,
        })
    except Exception as exc:
        return jsonify({"success": False, "error": str(exc)}), 500


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)

## CELL 33 — `requirements.txt` for the Backend

The minimal dependency set the Flask backend needs to run outside Colab. Pin to whatever versions Colab actually used if you want byte-identical behavior (`!pip show scikit-learn xgboost lightgbm catboost` after CELL 1 will show you the installed versions).

In [ ]:
%%writefile requirements.txt
flask
pandas
numpy
scikit-learn
joblib
xgboost
lightgbm
catboost

## CELL 34 — React Frontend Integration Example

Not Python — this is reference code for your React app (not meant to be run inside this notebook). It calls `POST /predict-credit-score`, shows a loading state, displays the score/category/interval, and handles errors.

```jsx
import { useState } from "react";

function CreditScoreForm() {
  const [formData, setFormData] = useState({
    Age: "", Gender: "", Education: "", Employment_Status: "", Occupation: "",
    Monthly_Income: "", Monthly_Expenses: "", Monthly_Savings: "", Current_Balance: "",
    Existing_Debt: "", Investment_Value: "", Insurance_Value: "", Credit_Utilization: "",
    Credit_History_Years: "", Outstanding_Debt: "", Credit_Cards: "", Credit_Inquiries: "",
    Payment_History: "", Credit_Mix: "", Failed_Transactions: "", Savings_Ratio: "",
    Income_Stability: "", Financial_Health_Score: "",
  });
  const [result, setResult] = useState(null);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState(null);

  const handleChange = (e) => {
    setFormData({ ...formData, [e.target.name]: e.target.value });
  };

  const handleSubmit = async (e) => {
    e.preventDefault();
    setLoading(true);
    setError(null);
    setResult(null);
    try {
      const response = await fetch("http://localhost:5000/predict-credit-score", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(formData),
      });
      const data = await response.json();
      if (!data.success) {
        setError(data.error || "Prediction failed");
      } else {
        setResult(data);
      }
    } catch (err) {
      setError("Could not reach the prediction service.");
    } finally {
      setLoading(false);
    }
  };

  return (
    <div>
      <form onSubmit={handleSubmit}>
        {/* render one input per field in formData, e.g.: */}
        <input name="Age" value={formData.Age} onChange={handleChange} placeholder="Age" />
        {/* ...remaining fields... */}
        <button type="submit" disabled={loading}>
          {loading ? "Predicting..." : "Predict Credit Score"}
        </button>
      </form>

      {error && <p style={{ color: "red" }}>{error}</p>}

      {result && (
        <div>
          <h3>Predicted Credit Score: {result.predicted_credit_score}</h3>
          <p>Range: {result.lower_bound} - {result.upper_bound}</p>
          <p>Category: {result.credit_score_category}</p>
        </div>
      )}
    </div>
  );
}

export default CreditScoreForm;
```

## Deployment Requirements

**Files to move from Colab into your application:**
- `credit_score_model.joblib` — downloaded in CELL 31.
- `app.py` — written in CELL 32 (copy it out of the Colab filesystem, or re-download it via `colab_files.download("app.py")`).
- `requirements.txt` — written in CELL 33.

**Where to place the model file:** in the same directory `app.py` runs from (or update the `joblib.load(...)` path in `app.py` to point wherever you store it, e.g. a `models/` subfolder).

**How the backend loads it:** once, at process startup, into the module-level `ARTIFACT`/`PIPELINE` variables in CELL 32 — every request reuses the same in-memory pipeline instead of reloading from disk.

**How to run locally:**
```
pip install -r requirements.txt
python app.py
```
This starts the Flask server on `http://localhost:5000`. Test it with:
```
curl -X POST http://localhost:5000/predict-credit-score -H "Content-Type: application/json" -d '{"Age": 35, "Gender": "Male", ...}'
```

**Deploying beyond localhost:** put `app.py` behind a production WSGI server (e.g. `gunicorn app:app`) and a process manager/container (Docker, systemd, or your platform's app-hosting service). No code changes are needed beyond that — the pipeline logic is identical.

## CELL 35 — Final End-to-End Test

One realistic sample customer, run through the complete path: raw input → feature engineering → preprocessing → numeric prediction → uncertainty interval → category → final JSON-like output — using the reloaded artifact from CELL 30, exactly as the Flask backend would.

In [ ]:
sample_customer = {
    "Age": 34,
    "Gender": "Female",
    "Education": "Bachelor's",
    "Employment_Status": "Salaried",
    "Occupation": "Software Engineer",
    "Monthly_Income": 85000.0,
    "Monthly_Expenses": 42000.0,
    "Monthly_Savings": 30000.0,
    "Current_Balance": 250000.0,
    "Existing_Debt": 60000.0,
    "Investment_Value": 400000.0,
    "Insurance_Value": 150000.0,
    "Credit_Utilization": 18.5,
    "Credit_History_Years": 9.0,
    "Outstanding_Debt": 25000.0,
    "Credit_Cards": 2,
    "Credit_Inquiries": 1,
    "Payment_History": "Good",
    "Credit_Mix": "Good",
    "Failed_Transactions": 0,
    "Savings_Ratio": 0.35,
    "Income_Stability": "Stable",
    "Financial_Health_Score": 62.0,
}

final_output = predict_from_artifact(RELOADED_ARTIFACT, sample_customer)
print("=== FINAL END-TO-END TEST ===")
print(json.dumps({"success": True, **final_output}, indent=2))

## COMPLETE PRODUCTION WORKFLOW

```
Dataset (CSV upload)
  -> Data Validation (missing values, duplicates, impossible values)
  -> Target Analysis (distribution, percentiles, outliers)
  -> Category Configuration (credit_score_to_category, editable boundaries)
  -> Leakage Analysis (Annual_Income/Expense_Ratio redundancy, Financial_Health_Score check,
                        Payment_History/Credit_Mix legitimacy, Customer_ID exclusion)
  -> Feature Engineering (5 documented ratio features, safe division)
  -> Exploratory Data Analysis (correlations, grouped stats)
  -> Train/Test Split (80/20, quantile-stratified, test set held out)
  -> Preprocessing Pipeline (median/mode imputation, scaling, one-hot encoding)
  -> Baseline Model (DummyRegressor)
  -> Model Comparison (8 algorithms, 5-fold CV, ranked by RMSE)
  -> Hyperparameter Optimization (top-3 candidates, RandomizedSearchCV)
  -> Automatic Best-Model Selection (lowest CV RMSE, overfit-gap aware)
  -> Final Unseen Test Evaluation (MAE/MSE/RMSE/R2/MAPE, ONE TIME)
  -> Reliability Analysis (% within +/-5/10/20/30 points)
  -> Feature Importance (permutation importance, top 20)
  -> Production Pipeline (predict_credit_score function)
  -> Uncertainty Estimation (OOF residual quantile interval)
  -> Save Model (single .joblib artifact: pipeline + config + metadata + metrics)
  -> Reload Test (proves the saved file works standalone)
  -> Download Model (Colab file download)
  -> Backend API (Flask app.py, loads once, POST /predict-credit-score)
  -> React Frontend (fetch example, loading/error states)
```

## HOW TO USE THE TRAINED MODEL IN MY APPLICATION

1. Run every cell in this notebook top to bottom in Google Colab (CELL 1 through CELL 35).
2. When CELL 31 runs, `credit_score_model.joblib` downloads through your browser.
3. Move that file into your backend project directory (next to `app.py`).
4. Copy `app.py` (CELL 32) and `requirements.txt` (CELL 33) into that same project directory.
5. Install dependencies: `pip install -r requirements.txt`.
6. Start the backend: `python app.py` — this loads the model once and keeps it in memory.
7. Send a customer's raw financial fields as JSON to `POST /predict-credit-score`.
8. Receive the numeric predicted credit score plus its `lower_bound`/`upper_bound` interval.
9. Convert/display the returned `credit_score_category` directly — it's already computed server-side.
10. Connect your React frontend (CELL 34) to call the same endpoint and render the result.
11. Test the full path end-to-end with a real sample customer, the same way CELL 35 does inside the notebook.
